In [1]:
# ============================================
# BIZMIND-AI
# SQL BUSINESS ANALYSIS
# ============================================

import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

print("SQL Analysis Started")

SQL Analysis Started


In [2]:
# ============================================
# LOAD CLEANED DATA
# ============================================

df = pd.read_csv("../data/processed/rfm_customer_segments.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Rows: 4322
Columns: 9


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,CustomerSegment
0,12347.0,2,7,4310.00,5,4,5,545,Champions
1,12348.0,75,4,1797.24,2,3,4,234,Needs Attention
2,12349.0,19,1,1757.55,4,1,4,414,New Customers
3,12350.0,310,1,334.40,1,1,2,112,Lost Customers
4,12352.0,36,11,1545.41,3,5,4,354,Potential Loyalists


In [3]:
# ============================================
# CREATE SQLITE DATABASE
# ============================================

conn = sqlite3.connect("../data/bizmind.db")

df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully!")
print("Table created: customers")

SQLite database created successfully!
Table created: customers


In [4]:
# ============================================
# CHECK SQL TABLE
# ============================================

query = """
SELECT *
FROM customers
LIMIT 10;
"""

sql_result = pd.read_sql_query(query, conn)

display(sql_result)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,CustomerSegment
0,12347.0,2,7,4310.00,5,4,5,545,Champions
1,12348.0,75,4,1797.24,2,3,4,234,Needs Attention
2,12349.0,19,1,1757.55,4,1,4,414,New Customers
3,12350.0,310,1,334.40,1,1,2,112,Lost Customers
4,12352.0,36,11,1545.41,3,5,4,354,Potential Loyalists
5,12353.0,204,1,89.00,1,1,1,111,Lost Customers
6,12354.0,232,1,1079.40,1,1,4,114,Cannot Lose Them
7,12355.0,214,1,459.40,1,1,2,112,Lost Customers
8,12356.0,23,3,2811.43,4,3,5,435,Loyal Customers
9,12357.0,33,1,6207.67,3,1,5,315,Needs Attention


In [5]:
# ============================================
# SQL BUSINESS ANALYSIS 1
# CUSTOMER OVERVIEW
# ============================================

query = """
SELECT
    COUNT(*) AS TotalCustomers,
    ROUND(AVG(Frequency), 2) AS AvgPurchaseFrequency,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue,
    ROUND(SUM(Monetary), 2) AS TotalCustomerRevenue
FROM customers;
"""

customer_overview = pd.read_sql_query(query, conn)

print("CUSTOMER OVERVIEW")
display(customer_overview)

CUSTOMER OVERVIEW


,TotalCustomers,AvgPurchaseFrequency,AvgCustomerValue,TotalCustomerRevenue
0,4322,5.11,1918.5,8291748.56


In [6]:
# ============================================
# SQL BUSINESS ANALYSIS 2
# CUSTOMER SEGMENT DISTRIBUTION
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue
FROM customers
GROUP BY CustomerSegment
ORDER BY TotalRevenue DESC;
"""

segment_analysis = pd.read_sql_query(query, conn)

print("CUSTOMER SEGMENT ANALYSIS")
display(segment_analysis)

CUSTOMER SEGMENT ANALYSIS


,CustomerSegment,Customers,TotalRevenue,AvgCustomerValue
0,Champions,946,5569196.13,5887.10
1,Potential Loyalists,497,891677.70,1794.12
2,At Risk,295,491382.70,1665.70
3,Needs Attention,750,424501.93,566.00
4,Loyal Customers,470,364104.66,774.69
5,Cannot Lose Them,248,236645.21,954.21
6,Lost Customers,793,178258.56,224.79
7,New Customers,323,135981.67,421.00


In [7]:
# ============================================
# SQL BUSINESS ANALYSIS 3
# TOP 10 CUSTOMERS BY REVENUE
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(Monetary, 2) AS TotalRevenue,
    Frequency,
    Recency,
    CustomerSegment
FROM customers
ORDER BY Monetary DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(query, conn)

print("TOP 10 CUSTOMERS BY REVENUE")
display(top_customers)


TOP 10 CUSTOMERS BY REVENUE


,CustomerID,TotalRevenue,Frequency,Recency,CustomerSegment
0,14646.0,279489.02,76,2,Champions
1,18102.0,256438.49,62,1,Champions
2,17450.0,187322.17,55,8,Champions
3,14911.0,132458.73,248,1,Champions
4,12415.0,123725.45,26,24,Champions
5,14156.0,113214.59,66,10,Champions
6,17511.0,88125.38,46,3,Champions
7,16684.0,65892.08,31,4,Champions
8,13694.0,62690.54,60,4,Champions
9,15311.0,59284.19,118,1,Champions


In [8]:
# ============================================
# SQL BUSINESS ANALYSIS 4
# TOP 10 CHAMPIONS BY REVENUE
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(Monetary, 2) AS TotalRevenue,
    Frequency,
    Recency
FROM customers
WHERE CustomerSegment = 'Champions'
ORDER BY Monetary DESC
LIMIT 10;
"""

champions = pd.read_sql_query(query, conn)

print("TOP 10 CHAMPIONS BY REVENUE")
display(champions)

TOP 10 CHAMPIONS BY REVENUE


,CustomerID,TotalRevenue,Frequency,Recency
0,14646.0,279489.02,76,2
1,18102.0,256438.49,62,1
2,17450.0,187322.17,55,8
3,14911.0,132458.73,248,1
4,12415.0,123725.45,26,24
5,14156.0,113214.59,66,10
6,17511.0,88125.38,46,3
7,16684.0,65892.08,31,4
8,13694.0,62690.54,60,4
9,15311.0,59284.19,118,1


In [9]:
# ============================================
# SQL BUSINESS ANALYSIS 5
# RFM REVENUE CONTRIBUTION
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(
        SUM(Monetary) * 100.0 /
        (SELECT SUM(Monetary) FROM customers),
        2
    ) AS RevenuePercentage
FROM customers
GROUP BY CustomerSegment
ORDER BY TotalRevenue DESC;
"""

rfm_revenue = pd.read_sql_query(query, conn)

print("RFM REVENUE CONTRIBUTION")
display(rfm_revenue)

RFM REVENUE CONTRIBUTION


,CustomerSegment,Customers,TotalRevenue,RevenuePercentage
0,Champions,946,5569196.13,67.17
1,Potential Loyalists,497,891677.70,10.75
2,At Risk,295,491382.70,5.93
3,Needs Attention,750,424501.93,5.12
4,Loyal Customers,470,364104.66,4.39
5,Cannot Lose Them,248,236645.21,2.85
6,Lost Customers,793,178258.56,2.15
7,New Customers,323,135981.67,1.64


In [10]:
# ============================================
# SQL BUSINESS ANALYSIS 6
# AT RISK CUSTOMERS
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(Monetary, 2) AS TotalRevenue,
    Frequency,
    Recency
FROM customers
WHERE CustomerSegment = 'At Risk'
ORDER BY Monetary DESC
LIMIT 20;
"""

at_risk = pd.read_sql_query(query, conn)

print("TOP 20 AT RISK CUSTOMERS")
display(at_risk)

TOP 20 AT RISK CUSTOMERS


,CustomerID,TotalRevenue,Frequency,Recency
0,15749.0,21535.90,4,235
1,12409.0,11056.93,7,79
2,16180.0,10217.48,10,100
3,13093.0,7741.47,13,267
4,16745.0,7157.10,18,87
5,12980.0,7092.06,12,156
6,13027.0,6912.00,6,114
7,16182.0,6617.65,4,72
8,15939.0,6102.26,16,90
9,14101.0,5976.79,6,74


In [11]:
# ============================================
# SQL BUSINESS ANALYSIS 7
# LOST CUSTOMERS
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(Monetary, 2) AS TotalRevenue,
    Frequency,
    Recency
FROM customers
WHERE CustomerSegment = 'Lost Customers'
ORDER BY Monetary DESC
LIMIT 20;
"""

lost_customers = pd.read_sql_query(query, conn)

print("TOP 20 LOST CUSTOMERS")
display(lost_customers)

TOP 20 LOST CUSTOMERS


,CustomerID,TotalRevenue,Frequency,Recency
0,12644.0,477.91,1,130
1,12447.0,476.49,1,243
2,15507.0,475.86,1,172
3,13043.0,471.57,2,254
4,14896.0,468.77,2,205
5,13248.0,465.68,2,125
6,16586.0,463.95,1,249
7,15349.0,463.91,2,158
8,15109.0,463.75,2,241
9,14489.0,463.38,1,207


In [12]:
# ============================================
# SQL BUSINESS ANALYSIS 8
# CUSTOMER SEGMENT PERFORMANCE
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue,
    ROUND(AVG(Frequency), 2) AS AvgFrequency,
    ROUND(AVG(Recency), 2) AS AvgRecency
FROM customers
GROUP BY CustomerSegment
ORDER BY TotalRevenue DESC;
"""

segment_performance = pd.read_sql_query(query, conn)

print("CUSTOMER SEGMENT PERFORMANCE")
display(segment_performance)

CUSTOMER SEGMENT PERFORMANCE


,CustomerSegment,Customers,TotalRevenue,AvgCustomerValue,AvgFrequency,AvgRecency
0,Champions,946,5569196.13,5887.10,13.55,11.63
1,Potential Loyalists,497,891677.70,1794.12,5.56,47.68
2,At Risk,295,491382.70,1665.70,5.91,134.22
3,Needs Attention,750,424501.93,566.00,1.95,109.87
4,Loyal Customers,470,364104.66,774.69,3.45,14.65
5,Cannot Lose Them,248,236645.21,954.21,1.43,178.39
6,Lost Customers,793,178258.56,224.79,1.14,223.36
7,New Customers,323,135981.67,421.00,1.37,17.11


In [13]:
# ============================================
# SQL BUSINESS ANALYSIS 9
# REVENUE CONTRIBUTION BY CUSTOMER SEGMENT
# ============================================

query = """
SELECT
    CustomerSegment,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(
        SUM(Monetary) * 100.0 /
        (SELECT SUM(Monetary) FROM customers),
        2
    ) AS RevenuePercentage
FROM customers
GROUP BY CustomerSegment
ORDER BY TotalRevenue DESC;
"""

revenue_contribution = pd.read_sql_query(query, conn)

print("REVENUE CONTRIBUTION BY CUSTOMER SEGMENT")
display(revenue_contribution)

REVENUE CONTRIBUTION BY CUSTOMER SEGMENT


,CustomerSegment,TotalRevenue,RevenuePercentage
0,Champions,5569196.13,67.17
1,Potential Loyalists,891677.70,10.75
2,At Risk,491382.70,5.93
3,Needs Attention,424501.93,5.12
4,Loyal Customers,364104.66,4.39
5,Cannot Lose Them,236645.21,2.85
6,Lost Customers,178258.56,2.15
7,New Customers,135981.67,1.64


In [14]:
# ============================================
# SQL BUSINESS ANALYSIS 10
# CUSTOMER RETENTION & RFM SEGMENT SUMMARY
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(AVG(Frequency), 2) AS AvgFrequency,
    ROUND(AVG(Recency), 2) AS AvgRecency,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue
FROM customers
GROUP BY CustomerSegment
ORDER BY AvgCustomerValue DESC;
"""

retention_analysis = pd.read_sql_query(query, conn)

print("CUSTOMER RETENTION & RFM SEGMENT SUMMARY")
display(retention_analysis)

CUSTOMER RETENTION & RFM SEGMENT SUMMARY


,CustomerSegment,Customers,AvgFrequency,AvgRecency,AvgCustomerValue
0,Champions,946,13.55,11.63,5887.10
1,Potential Loyalists,497,5.56,47.68,1794.12
2,At Risk,295,5.91,134.22,1665.70
3,Cannot Lose Them,248,1.43,178.39,954.21
4,Loyal Customers,470,3.45,14.65,774.69
5,Needs Attention,750,1.95,109.87,566.00
6,New Customers,323,1.37,17.11,421.00
7,Lost Customers,793,1.14,223.36,224.79


In [15]:
# ============================================
# SQL BUSINESS ANALYSIS 11
# TOP 10 CUSTOMERS BY REVENUE
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    SUM(Frequency) AS TotalOrders,
    MIN(Recency) AS Recency,
    CustomerSegment
FROM customers
GROUP BY CustomerID, CustomerSegment
ORDER BY TotalRevenue DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(query, conn)

print("TOP 10 CUSTOMERS BY REVENUE")
display(top_customers)

TOP 10 CUSTOMERS BY REVENUE


,CustomerID,TotalRevenue,TotalOrders,Recency,CustomerSegment
0,14646.0,279489.02,76,2,Champions
1,18102.0,256438.49,62,1,Champions
2,17450.0,187322.17,55,8,Champions
3,14911.0,132458.73,248,1,Champions
4,12415.0,123725.45,26,24,Champions
5,14156.0,113214.59,66,10,Champions
6,17511.0,88125.38,46,3,Champions
7,16684.0,65892.08,31,4,Champions
8,13694.0,62690.54,60,4,Champions
9,15311.0,59284.19,118,1,Champions


In [16]:
# ============================================
# SQL BUSINESS ANALYSIS 12
# CUSTOMER SEGMENT REVENUE RANKING
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue
FROM customers
GROUP BY CustomerSegment
ORDER BY TotalRevenue DESC;
"""

segment_revenue = pd.read_sql_query(query, conn)

print("CUSTOMER SEGMENT REVENUE RANKING")
display(segment_revenue)

CUSTOMER SEGMENT REVENUE RANKING


,CustomerSegment,Customers,TotalRevenue,AvgCustomerValue
0,Champions,946,5569196.13,5887.10
1,Potential Loyalists,497,891677.70,1794.12
2,At Risk,295,491382.70,1665.70
3,Needs Attention,750,424501.93,566.00
4,Loyal Customers,470,364104.66,774.69
5,Cannot Lose Them,248,236645.21,954.21
6,Lost Customers,793,178258.56,224.79
7,New Customers,323,135981.67,421.00


In [17]:
# ============================================
# SQL BUSINESS ANALYSIS 13
# TOP 10 CUSTOMERS BY PURCHASE FREQUENCY
# ============================================

query = """
SELECT
    CustomerID,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    Recency,
    CustomerSegment
FROM customers
ORDER BY Frequency DESC
LIMIT 10;
"""

top_frequency_customers = pd.read_sql_query(query, conn)

print("TOP 10 CUSTOMERS BY PURCHASE FREQUENCY")
display(top_frequency_customers)

TOP 10 CUSTOMERS BY PURCHASE FREQUENCY


,CustomerID,Frequency,TotalRevenue,Recency,CustomerSegment
0,14911.0,248,132458.73,1,Champions
1,12748.0,223,28405.56,1,Champions
2,17841.0,169,39869.05,2,Champions
3,14606.0,128,11633.35,1,Champions
4,13089.0,118,57322.13,3,Champions
5,15311.0,118,59284.19,1,Champions
6,12971.0,89,10930.26,4,Champions
7,14527.0,86,7709.69,3,Champions
8,13408.0,81,27487.41,2,Champions
9,14646.0,76,279489.02,2,Champions


In [18]:
# ============================================
# SQL BUSINESS ANALYSIS 14
# TOP 10 CUSTOMERS BY MONETARY VALUE
# ============================================

query = """
SELECT
    CustomerID,
    ROUND(Monetary, 2) AS TotalRevenue,
    Frequency,
    Recency,
    CustomerSegment
FROM customers
ORDER BY Monetary DESC
LIMIT 10;
"""

top_monetary_customers = pd.read_sql_query(query, conn)

print("TOP 10 CUSTOMERS BY MONETARY VALUE")
display(top_monetary_customers)

TOP 10 CUSTOMERS BY MONETARY VALUE


,CustomerID,TotalRevenue,Frequency,Recency,CustomerSegment
0,14646.0,279489.02,76,2,Champions
1,18102.0,256438.49,62,1,Champions
2,17450.0,187322.17,55,8,Champions
3,14911.0,132458.73,248,1,Champions
4,12415.0,123725.45,26,24,Champions
5,14156.0,113214.59,66,10,Champions
6,17511.0,88125.38,46,3,Champions
7,16684.0,65892.08,31,4,Champions
8,13694.0,62690.54,60,4,Champions
9,15311.0,59284.19,118,1,Champions


In [19]:
# ============================================
# SQL BUSINESS ANALYSIS 15
# TOP 10 MOST RECENT CUSTOMERS
# ============================================

query = """
SELECT
    CustomerID,
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    CustomerSegment
FROM customers
ORDER BY Recency ASC
LIMIT 10;
"""

recent_customers = pd.read_sql_query(query, conn)

print("TOP 10 MOST RECENT CUSTOMERS")
display(recent_customers)

TOP 10 MOST RECENT CUSTOMERS


,CustomerID,Recency,Frequency,TotalRevenue,CustomerSegment
0,12423.0,1,9,1849.11,Champions
1,12433.0,1,7,13375.87,Champions
2,12476.0,1,20,6546.58,Champions
3,12518.0,1,5,2056.89,Champions
4,12526.0,1,3,1316.66,Loyal Customers
5,12662.0,1,12,3817.08,Champions
6,12680.0,1,4,862.81,Loyal Customers
7,12713.0,1,1,848.55,New Customers
8,12748.0,1,223,28405.56,Champions
9,12955.0,1,14,4734.26,Champions


In [20]:
# ============================================
# SQL BUSINESS ANALYSIS 16
# CUSTOMERS AT RISK OF CHURNING
# ============================================

query = """
SELECT
    CustomerID,
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    CustomerSegment
FROM customers
WHERE CustomerSegment IN ('At Risk', 'Cannot Lose Them', 'Lost Customers')
ORDER BY Recency DESC
LIMIT 20;
"""

churn_risk_customers = pd.read_sql_query(query, conn)

print("CUSTOMERS AT RISK OF CHURNING")
display(churn_risk_customers)

CUSTOMERS AT RISK OF CHURNING


,CustomerID,Recency,Frequency,TotalRevenue,CustomerSegment
0,12791.0,374,1,192.60,Lost Customers
1,13747.0,374,1,79.60,Lost Customers
2,14729.0,374,1,313.49,Lost Customers
3,16583.0,374,1,233.45,Lost Customers
4,17908.0,374,1,232.03,Lost Customers
5,17968.0,374,1,265.10,Lost Customers
6,18074.0,374,1,489.60,Cannot Lose Them
7,12855.0,373,1,38.10,Lost Customers
8,13065.0,373,1,205.86,Lost Customers
9,13108.0,373,1,350.06,Lost Customers


In [21]:
# ============================================
# SQL BUSINESS ANALYSIS 17
# HIGH-VALUE CUSTOMERS AT RISK
# ============================================

query = """
SELECT
    CustomerID,
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    CustomerSegment
FROM customers
WHERE CustomerSegment IN ('At Risk', 'Cannot Lose Them')
  AND Monetary > (
      SELECT AVG(Monetary)
      FROM customers
  )
ORDER BY Monetary DESC
LIMIT 20;
"""

high_value_at_risk = pd.read_sql_query(query, conn)

print("HIGH-VALUE CUSTOMERS AT RISK")
display(high_value_at_risk)

HIGH-VALUE CUSTOMERS AT RISK


,CustomerID,Recency,Frequency,TotalRevenue,CustomerSegment
0,15749.0,235,4,21535.90,At Risk
1,12409.0,79,7,11056.93,At Risk
2,16180.0,100,10,10217.48,At Risk
3,12435.0,80,2,7829.89,Cannot Lose Them
4,13093.0,267,13,7741.47,At Risk
5,16745.0,87,18,7157.10,At Risk
6,12980.0,156,12,7092.06,At Risk
7,13027.0,114,6,6912.00,At Risk
8,16182.0,72,4,6617.65,At Risk
9,15939.0,90,16,6102.26,At Risk


In [22]:
# ============================================
# SQL BUSINESS ANALYSIS 18
# CHURN RISK BY CUSTOMER SEGMENT
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS CustomersAtRisk,
    ROUND(SUM(Monetary), 2) AS AtRiskRevenue,
    ROUND(AVG(Recency), 2) AS AvgRecency,
    ROUND(AVG(Frequency), 2) AS AvgFrequency
FROM customers
WHERE CustomerSegment IN ('At Risk', 'Cannot Lose Them', 'Lost Customers')
GROUP BY CustomerSegment
ORDER BY AtRiskRevenue DESC;
"""

churn_summary = pd.read_sql_query(query, conn)

print("CHURN RISK BY CUSTOMER SEGMENT")
display(churn_summary)

CHURN RISK BY CUSTOMER SEGMENT


,CustomerSegment,CustomersAtRisk,AtRiskRevenue,AvgRecency,AvgFrequency
0,At Risk,295,491382.70,134.22,5.91
1,Cannot Lose Them,248,236645.21,178.39,1.43
2,Lost Customers,793,178258.56,223.36,1.14


In [23]:
# ============================================
# SQL BUSINESS ANALYSIS 19
# CUSTOMER VALUE CLASSIFICATION
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(
        SUM(Monetary) * 100.0 /
        (SELECT SUM(Monetary) FROM customers),
        2
    ) AS RevenuePercentage
FROM customers
GROUP BY CustomerSegment
ORDER BY AvgCustomerValue DESC;
"""

customer_value = pd.read_sql_query(query, conn)

print("CUSTOMER VALUE CLASSIFICATION")
display(customer_value)


CUSTOMER VALUE CLASSIFICATION


,CustomerSegment,Customers,AvgCustomerValue,TotalRevenue,RevenuePercentage
0,Champions,946,5887.10,5569196.13,67.17
1,Potential Loyalists,497,1794.12,891677.70,10.75
2,At Risk,295,1665.70,491382.70,5.93
3,Cannot Lose Them,248,954.21,236645.21,2.85
4,Loyal Customers,470,774.69,364104.66,4.39
5,Needs Attention,750,566.00,424501.93,5.12
6,New Customers,323,421.00,135981.67,1.64
7,Lost Customers,793,224.79,178258.56,2.15


In [24]:
# ============================================
# SQL BUSINESS ANALYSIS 20
# REPEAT VS ONE-TIME CUSTOMER ANALYSIS
# ============================================

query = """
SELECT
    CASE
        WHEN Frequency > 1 THEN 'Repeat Customer'
        ELSE 'One-Time Customer'
    END AS CustomerType,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue,
    ROUND(AVG(Frequency), 2) AS AvgFrequency
FROM customers
GROUP BY
    CASE
        WHEN Frequency > 1 THEN 'Repeat Customer'
        ELSE 'One-Time Customer'
    END
ORDER BY TotalRevenue DESC;
"""

repeat_customer_analysis = pd.read_sql_query(query, conn)

print("REPEAT VS ONE-TIME CUSTOMER ANALYSIS")
display(repeat_customer_analysis)

REPEAT VS ONE-TIME CUSTOMER ANALYSIS


,CustomerType,Customers,TotalRevenue,AvgCustomerValue,AvgFrequency
0,Repeat Customer,3040,7849801.96,2582.17,6.85
1,One-Time Customer,1282,441946.60,344.73,1.00


In [25]:
# ============================================
# SQL BUSINESS ANALYSIS 21
# TOP 10% HIGH-VALUE CUSTOMER CONTRIBUTION
# ============================================

query = """
WITH ranked_customers AS (
    SELECT
        CustomerID,
        Monetary,
        Frequency,
        Recency,
        CustomerSegment,
        NTILE(10) OVER (ORDER BY Monetary DESC) AS RevenueDecile
    FROM customers
)

SELECT
    CASE
        WHEN RevenueDecile = 1 THEN 'Top 10%'
        ELSE 'Other 90%'
    END AS CustomerGroup,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue,
    ROUND(AVG(Frequency), 2) AS AvgFrequency
FROM ranked_customers
GROUP BY
    CASE
        WHEN RevenueDecile = 1 THEN 'Top 10%'
        ELSE 'Other 90%'
    END
ORDER BY TotalRevenue DESC;
"""

top_10_contribution = pd.read_sql_query(query, conn)

print("TOP 10% HIGH-VALUE CUSTOMER CONTRIBUTION")
display(top_10_contribution)

TOP 10% HIGH-VALUE CUSTOMER CONTRIBUTION


,CustomerGroup,Customers,TotalRevenue,AvgCustomerValue,AvgFrequency
0,Top 10%,433,4961063.12,11457.42,20.38
1,Other 90%,3889,3330685.44,856.44,3.42


In [26]:
# ============================================
# SQL BUSINESS ANALYSIS 22
# CUSTOMER SEGMENT DISTRIBUTION
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customers),
        2
    ) AS CustomerPercentage
FROM customers
GROUP BY CustomerSegment
ORDER BY Customers DESC;
"""

segment_distribution = pd.read_sql_query(query, conn)

print("CUSTOMER SEGMENT DISTRIBUTION")
display(segment_distribution)

CUSTOMER SEGMENT DISTRIBUTION


,CustomerSegment,Customers,CustomerPercentage
0,Champions,946,21.89
1,Lost Customers,793,18.35
2,Needs Attention,750,17.35
3,Potential Loyalists,497,11.50
4,Loyal Customers,470,10.87
5,New Customers,323,7.47
6,At Risk,295,6.83
7,Cannot Lose Them,248,5.74


In [27]:
# ============================================
# SQL BUSINESS ANALYSIS 23
# SEGMENT REVENUE PER CUSTOMER
# ============================================

query = """
SELECT
    CustomerSegment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(SUM(Monetary) / COUNT(*), 2) AS RevenuePerCustomer
FROM customers
GROUP BY CustomerSegment
ORDER BY RevenuePerCustomer DESC;
"""

segment_value = pd.read_sql_query(query, conn)

print("SEGMENT REVENUE PER CUSTOMER")
display(segment_value)

SEGMENT REVENUE PER CUSTOMER


,CustomerSegment,Customers,TotalRevenue,RevenuePerCustomer
0,Champions,946,5569196.13,5887.10
1,Potential Loyalists,497,891677.70,1794.12
2,At Risk,295,491382.70,1665.70
3,Cannot Lose Them,248,236645.21,954.21
4,Loyal Customers,470,364104.66,774.69
5,Needs Attention,750,424501.93,566.00
6,New Customers,323,135981.67,421.00
7,Lost Customers,793,178258.56,224.79


In [28]:
# ============================================
# SQL BUSINESS ANALYSIS 24
# CUSTOMER VALUE VS PURCHASE FREQUENCY
# ============================================

query = """
SELECT
    CustomerID,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    Recency,
    CustomerSegment
FROM customers
WHERE Frequency >= (
    SELECT AVG(Frequency)
    FROM customers
)
AND Monetary >= (
    SELECT AVG(Monetary)
    FROM customers
)
ORDER BY Monetary DESC
LIMIT 20;
"""

high_value_frequent = pd.read_sql_query(query, conn)

print("HIGH-VALUE & HIGH-FREQUENCY CUSTOMERS")
display(high_value_frequent)

HIGH-VALUE & HIGH-FREQUENCY CUSTOMERS


,CustomerID,Frequency,TotalRevenue,Recency,CustomerSegment
0,14646.0,76,279489.02,2,Champions
1,18102.0,62,256438.49,1,Champions
2,17450.0,55,187322.17,8,Champions
3,14911.0,248,132458.73,1,Champions
4,12415.0,26,123725.45,24,Champions
5,14156.0,66,113214.59,10,Champions
6,17511.0,46,88125.38,3,Champions
7,16684.0,31,65892.08,4,Champions
8,13694.0,60,62690.54,4,Champions
9,15311.0,118,59284.19,1,Champions


In [29]:
# ============================================
# SQL BUSINESS ANALYSIS 25
# RFM SCORE DISTRIBUTION
# ============================================

query = """
SELECT
    RFM_Score,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS TotalRevenue,
    ROUND(AVG(Monetary), 2) AS AvgCustomerValue
FROM customers
GROUP BY RFM_Score
ORDER BY TotalRevenue DESC;
"""

rfm_score_distribution = pd.read_sql_query(query, conn)

print("RFM SCORE DISTRIBUTION")
display(rfm_score_distribution)

RFM SCORE DISTRIBUTION


,RFM_Score,Customers,TotalRevenue,AvgCustomerValue
0,555,341,3790611.45,11116.16
1,455,170,980183.41,5765.78
2,355,78,371673.51,4765.05
3,545,51,173912.51,3410.05
4,345,47,150299.49,3197.86
...,...,...,...,...
113,552,1,389.86,389.86
114,251,1,230.70,230.70
115,141,1,210.32,210.32
116,451,1,191.17,191.17


In [30]:
# ============================================
# SQL BUSINESS ANALYSIS 26
# RFM SCORE SUMMARY
# ============================================

query = """
SELECT
    RFM_Score,
    COUNT(*) AS Customers,
    ROUND(AVG(Recency), 2) AS AvgRecency,
    ROUND(AVG(Frequency), 2) AS AvgFrequency,
    ROUND(AVG(Monetary), 2) AS AvgMonetary
FROM customers
GROUP BY RFM_Score
ORDER BY RFM_Score DESC;
"""

rfm_score_summary = pd.read_sql_query(query, conn)

print("RFM SCORE SUMMARY")
display(rfm_score_summary)

RFM SCORE SUMMARY


,RFM_Score,Customers,AvgRecency,AvgFrequency,AvgMonetary
0,555,341,5.00,22.41,11116.16
1,554,77,5.31,9.52,1527.21
2,553,5,3.00,7.80,648.19
3,552,1,3.00,7.00,389.86
4,545,51,6.22,5.49,3410.05
...,...,...,...,...,...
113,115,2,258.50,1.00,2878.62
114,114,15,267.13,1.00,1260.10
115,113,39,272.62,1.00,616.87
116,112,108,268.39,1.00,340.93


In [31]:
# ============================================
# SQL BUSINESS ANALYSIS 27
# FINAL CUSTOMER INTELLIGENCE DATASET
# ============================================

query = """
SELECT
    CustomerID,
    Recency,
    Frequency,
    ROUND(Monetary, 2) AS TotalRevenue,
    R_Score,
    F_Score,
    M_Score,
    RFM_Score,
    CustomerSegment
FROM customers
ORDER BY Monetary DESC;
"""

customer_intelligence = pd.read_sql_query(query, conn)

print("FINAL CUSTOMER INTELLIGENCE DATASET")
print("Rows:", len(customer_intelligence))
print("Columns:", len(customer_intelligence.columns))

display(customer_intelligence.head(20))

FINAL CUSTOMER INTELLIGENCE DATASET
Rows: 4322
Columns: 9


,CustomerID,Recency,Frequency,TotalRevenue,R_Score,F_Score,M_Score,RFM_Score,CustomerSegment
0,14646.0,2,76,279489.02,5,5,5,555,Champions
1,18102.0,1,62,256438.49,5,5,5,555,Champions
2,17450.0,8,55,187322.17,5,5,5,555,Champions
3,14911.0,1,248,132458.73,5,5,5,555,Champions
4,12415.0,24,26,123725.45,4,5,5,455,Champions
5,14156.0,10,66,113214.59,5,5,5,555,Champions
6,17511.0,3,46,88125.38,5,5,5,555,Champions
7,16684.0,4,31,65892.08,5,5,5,555,Champions
8,13694.0,4,60,62690.54,5,5,5,555,Champions
9,15311.0,1,118,59284.19,5,5,5,555,Champions


In [32]:
# Export final SQL analysis dataset

customer_intelligence.to_csv(
    "../data/processed/customer_intelligence.csv",
    index=False
)

print("customer_intelligence.csv exported successfully!")

customer_intelligence.csv exported successfully!


In [33]:
# ============================================
# POWER BI DASHBOARD DATASET
# ============================================

dashboard_df = customer_intelligence.copy()

dashboard_df.to_csv(
    "../data/processed/bizmind_customer_dashboard.csv",
    index=False
)

print("Dashboard dataset created successfully!")
print("Rows:", len(dashboard_df))
print("Columns:", len(dashboard_df.columns))

Dashboard dataset created successfully!
Rows: 4322
Columns: 9
